In [1]:
import pandas as pd #pandas : manipulation de tableaux de données (DataFrames)
import numpy as np #numpy : calcul numérique
import matplotlib.pyplot as plt
import seaborn as sns #matplotlib et seaborn : visualisation (importés mais pas utilisés dans ce notebook)
from sklearn.impute  import  KNNImputer #KNNImputer : remplace les valeurs manquantes à partir des plus proches voisins

In [2]:
df_train = pd.read_csv('train.csv') #Lecture des fichiers d’entraînement et de test
df_test = pd.read_csv('test.csv') 
n_train = len(df_train) #nombre de lignes d'entraînement : sert à re-séparer train/test après le prétraitement
test_ids = df_test['PassengerId'] #on garde les PassengerId du test (utiles pour une soumission)
df_test['Transported'] = False #Le test n’a pas de colonne Transported. On en crée une fictive (False) pour que les deux tableaux aient les mêmes colonnes
df = pd.concat([df_train, df_test], sort = False) #pd.concat les empile en un seul DataFrame, ce qui permet d’appliquer le même prétraitement aux deux jeux
df.drop(['Name' , 'PassengerId'], axis = 1, inplace = True) #On supprime Name et PassengerId, jugés inutiles pour la prédiction
df.head() #df.head() affiche les 5 premières lignes

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


In [3]:
df.shape


(12970, 12)

In [4]:
df.isna().sum() #Compte les valeurs manquantes (NaN) par colonne pour savoir quoi nettoyer

HomePlanet      288
CryoSleep       310
Cabin           299
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Transported       0
dtype: int64

In [5]:
df[['Deck', 'Num' , 'Side']] = df['Cabin'].str.split('/', expand = True)
# La cabine a le format Pont/Numéro/Côté (ex. B/0/P). On la sépare en trois colonnes :
# Deck : le pont (A, B, C)
# Num : le numéro de cabine
# Side : le côté, P (bâbord) ou S (tribord)
df = df.drop(columns = ['Cabin'])
# La colonne Cabin d’origine est ensuite supprimée

In [6]:
df.head()

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Deck,Num,Side
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False,B,0,P
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True,F,0,S
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False,A,0,S
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False,A,0,S
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True,F,1,S


In [7]:
df['Deck'] = df['Deck'].fillna('U')
df['Num'] = df['Num'].fillna('-1').astype(float) #Num doit être numérique (c'était une chaîne)
df['Side'] = df['Side'].fillna('U')
df.isna().sum()
# Les valeurs manquantes reçoivent une valeur « inconnue » : 'U' (Unknown) pour Deck et Side, '-1' pour Num. Puis on revérifie les NaN

HomePlanet      288
CryoSleep       310
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Transported       0
Deck              0
Num               0
Side              0
dtype: int64

In [8]:
df['Deck'].value_counts()
# Affiche le nombre de passagers par pont. Cet ordre de fréquence sert à l’encodage du bloc suivant

Deck
F    4239
G    3781
E    1323
B    1141
C    1102
D     720
A     354
U     299
T      11
Name: count, dtype: int64

In [9]:
df['Deck'] = df['Deck'].map({'F': 0, 'G': 1, 'E': 2, 'B': 3, 'C': 4, 'D': 5, 'A': 6, 'U': 7, 'T': 8})
df['Side'] = df['Side'].map({'U': -1, 'P': 1, 'S': 2})
# Les modèles ont besoin de nombres. Chaque lettre est remplacée par un entier (label encoding) selon un dictionnaire

In [10]:
impute_lis = ['Age' , 'VIP' , 'Num' , 'CryoSleep', 'Side' , 'Deck' ,'RoomService' , 'FoodCourt' , 'ShoppingMall' , 'Spa' , 'VRDeck']
# impute_lis : colonnes à imputer
df[['VIP', 'CryoSleep']] = df[['VIP', 'CryoSleep']].astype(float) #True/False/NaN -> 1.0/0.0/NaN pour KNNImputer
rest = [c for c in df.columns if c not in impute_lis] # toutes les autres colonnes
df_rest = df[rest]
imp = KNNImputer(n_neighbors = 5) # KNNImputer(n_neighbors=5) remplace chaque NaN par la moyenne des valeurs des 5 passagers les plus proches (selon les autres colonnes)
df_imputed = imp.fit_transform(df[impute_lis]) # fit_transform retourne un tableau NumPy, reconverti en DataFrame
df_imputed = pd.DataFrame(df_imputed, columns = impute_lis)
df = pd.concat([df_rest.reset_index(drop = True), df_imputed.reset_index(drop = True)], axis = 1)
# reset_index(drop=True) réaligne les index (après le concat initial, ils sont dupliqués), puis concat(axis=1) recolle les deux morceaux


In [11]:
df['HomePlanet'] = df['HomePlanet'].fillna('U')
df['Destination'] = df['Destination'].fillna('U')
category_colls = ['HomePlanet', 'Destination']
for col in category_colls:
    df = pd.concat([df, pd.get_dummies(df[col], prefix = col)], axis = 1)

df = df.drop(columns = category_colls)

In [12]:
# from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
# from sklearn.model_selection import StratifiedKFold, cross_val_score
# from sklearn.preprocessing import OrdinalEncoder

# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# cols = ["HomePlanet", "Destination"]
# y = df_train["Transported"].astype(int)
# X_base = df.iloc[:len(df_train)].drop(columns=["Transported"], errors="ignore")

# # 1. Ordinal
# X_ord = X_base.copy()
# X_ord[cols] = OrdinalEncoder().fit_transform(X_ord[cols].fillna("U").astype(str))

# # 2. Native Category
# X_cat = X_base.copy()
# for c in cols:
#     X_cat[c] = X_cat[c].fillna("U").astype("category")

# # 3. Frequency
# X_freq = X_base.copy()
# for c in cols:
#     s = X_freq[c].fillna("U").astype(str)
#     X_freq[c] = s.map(s.value_counts(normalize=True))

# for name, X_cur, is_cat in [
#     ("Ordinal", X_ord, False),
#     ("NativeCat", X_cat, True),
#     ("Freq", X_freq, False),
# ]:
#     rf = "-" if is_cat else round(cross_val_score(RandomForestClassifier(random_state=42), X_cur, y, cv=cv).mean(), 4)
#     hgb = round(
#         cross_val_score(
#             HistGradientBoostingClassifier(categorical_features=cols if is_cat else None, random_state=42),
#             X_cur, y, cv=cv
#         ).mean(),
#         4,
#     )
#     print(f"{name}: RF={rf}, HGB={hgb}")

# feacture engineering


In [13]:
bill_cols = ['RoomService' , 'FoodCourt' , 'ShoppingMall' , 'Spa' , 'VRDeck']
df['amt_spent'] = df[bill_cols].sum(axis = 1)
df['std_amt_spent'] = df[bill_cols].std(axis = 1)
df['mean_amt_spent'] = df[bill_cols].mean(axis = 1)
df['3_high_cols'] = df['CryoSleep'] + df['HomePlanet_Earth'] + df['Destination_55 Cancri e']
df['3_low_cols'] = df['mean_amt_spent'] + df['amt_spent'] + df['HomePlanet_Earth']

In [14]:
df.head()

,Transported,Age,VIP,Num,CryoSleep,Side,Deck,RoomService,FoodCourt,ShoppingMall,...,HomePlanet_U,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Destination_U,amt_spent,std_amt_spent,mean_amt_spent,3_high_cols,3_low_cols
0,False,39.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,...,False,False,False,True,False,0.0,0.000000,0.0,0.0,0.0
1,True,24.0,0.0,0.0,0.0,2.0,0.0,109.0,9.0,25.0,...,False,False,False,True,False,736.0,227.807375,147.2,1.0,884.2
2,False,58.0,1.0,0.0,0.0,2.0,6.0,43.0,3576.0,0.0,...,False,False,False,True,False,10383.0,3013.383198,2076.6,0.0,12459.6
3,False,33.0,0.0,0.0,0.0,2.0,6.0,0.0,1283.0,371.0,...,False,False,False,True,False,5176.0,1373.410427,1035.2,0.0,6211.2
4,True,16.0,0.0,1.0,0.0,2.0,0.0,303.0,70.0,151.0,...,False,False,False,True,False,1091.0,223.988169,218.2,1.0,1310.2


In [15]:
df_tr = df.iloc[:n_train].copy() #corrélation sur le train uniquement (le test a une fausse cible)
df_tr['Transported'] = df_tr['Transported'].astype(int)
df_tr.corr()['Transported'].sort_values(ascending = False)
# ordre décroissant

Transported                  1.000000
CryoSleep                    0.465118
3_high_cols                  0.244755
HomePlanet_Europa            0.176916
Destination_55 Cancri e      0.108722
Side                         0.081955
Deck                         0.079749
FoodCourt                    0.044230
HomePlanet_Mars              0.019544
ShoppingMall                 0.007895
HomePlanet_U                 0.002712
Destination_U                0.000547
Destination_PSO J318.5-22    0.000092
VIP                         -0.037842
Num                         -0.043830
Age                         -0.074901
Destination_TRAPPIST-1e     -0.094700
HomePlanet_Earth            -0.169019
std_amt_spent               -0.173283
mean_amt_spent              -0.201619
amt_spent                   -0.201619
3_low_cols                  -0.201652
VRDeck                      -0.206184
Spa                         -0.219944
RoomService                 -0.242388
Name: Transported, dtype: float64

In [16]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# On utilise le DataFrame prétraité df (pas df_train brut) : seules les n_train premières lignes ont une vraie cible
train_df = df.iloc[:n_train]
x = train_df.drop(columns=["Transported"])
y = train_df["Transported"].astype(int)

variants = {
    "with 3_high_cols and 3_low_cols": x,
    "no_high_cols": x.drop(columns=["3_high_cols"]),
    "no_low_cols": x.drop(columns=["3_low_cols"]),
    "without both of them": x.drop(columns=["3_high_cols", "3_low_cols"]),
}

for label, X_sub in variants.items():
    rf_acc = cross_val_score(
        RandomForestClassifier(n_estimators=200, random_state=42),
        X_sub, y, cv=cv, scoring="accuracy"
    ).mean()
    hgb_acc = cross_val_score(
        HistGradientBoostingClassifier(random_state=42),
        X_sub, y, cv=cv, scoring="accuracy"
    ).mean()
    print(f"{label:34s} | RF: {rf_acc:.5f} | HistGB: {hgb_acc:.5f}")


with 3_high_cols and 3_low_cols    | RF: 0.79478 | HistGB: 0.81031
no_high_cols                       | RF: 0.79731 | HistGB: 0.80904
no_low_cols                        | RF: 0.79846 | HistGB: 0.80916
without both of them               | RF: 0.79800 | HistGB: 0.80824


In [17]:
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [20]:
x = df.iloc[:n_train].drop(columns=["Transported"])
y = df.iloc[:n_train]["Transported"].astype(int)

x_train, x_val, y_train, y_val = train_test_split(
    x, y, test_size=0.2, random_state=42
)

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
}

scores = {}
fitted_models = {}

for name, clf in models.items():
    clf.fit(x_train, y_train)
    preds = clf.predict(x_val)
    acc = accuracy_score(y_val, preds)
    scores[name] = acc
    fitted_models[name] = clf
    print(f"{name:20s}: {acc:.5f}")

best_model_name = max(scores, key=scores.get)
best_score = scores[best_model_name]
best_model = fitted_models[best_model_name]


print("Best model:", best_model_name ,"Accuracy: ",best_score)


c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression  : 0.77976
DecisionTree        : 0.72915
RandomForest        : 0.79068
XGBoost             : 0.79701
LightGBM            : 0.80161
Best model: LightGBM Accuracy:  0.8016101207590569
